In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [40]:
def remove_consecutive_duplicates(numbers):
    """Remove consecutive duplicates while maintaining original order"""
    # Initialize the result list
    result = []
    
    # Iterate through the list
    for i in range(len(numbers)):
        # Add the number to the result if it's the first element or different from the previous one
        if i == 0 or numbers[i] != numbers[i - 1]:
            result.append(numbers[i])
    
    return result

def count_plus30(templist):
    """Given a list of numbers (~temperatures), compute how many exceed 30"""
    res = 0
    for temp in templist:
        if temp >= 30:
            res += 1
    return res

def find_consecutive_highs(data, threshold=25.0, min_count=5):
    """ Find sequences of numbers exceeding a certain treshold
    Parameters
    ----------
    data
        the list or array containing the full sequence of numbers (~temperatures)
    treshold
        a float indicating the minimal value for a number in the sequence to be considered high.
        default is 25.0
    min_count 
        integer, minimal length of a sequence of consecutive highs. any sequence of highs that is not at least this length is discarded.
        default is 5
    Returns
    -------
        a list of tuples (start, end) indicating the index range of each of the high sequences
    """
    # List to store the start and end indices of consecutive sequences
    indices = []
    
    start_index = None  # Variable to track the start of a sequence
    current_count = 0   # Counter for consecutive numbers above threshold
    
    for i in range(len(data)):
        if data[i] >= threshold:
            # Start a new sequence if it's not already started
            if start_index is None:
                start_index = i
            current_count += 1
        else:
            # If a sequence was ongoing and it meets the minimum count, store it
            if start_index is not None and current_count >= min_count:
                if count_plus30(data[start_index: i-1]) >= 3:
                    indices.append((start_index, i - 1))
            
            # Reset the sequence
            start_index = None
            current_count = 0
    
    # Check if there was an ongoing sequence at the end of the list
    if start_index is not None and current_count >= min_count:
        if count_plus30(data[start_index: len(data)-1]) >= 3:
            indices.append((start_index, len(data) - 1))
    
    return indices

In [3]:
class Heatwave:
    """class used to represent a heatwave (using the Belgian RMI definition)

    Attributes
    ----------
    dates
        a list of the dates (tuples (y, m, d)) during which the heatwave is active
    maxtemps, mintemps
        lists (chronological) of daily TX and TN values resp.
    year
        integer representing the year in which the heatwave takes place
    hw_dict
        dictionary (tuple -> float) containing date tuples (y,m,d) as keys and resp. TX as values
    length
        integer representing the length of the heatwave in nr. of days
    tropic_day_count, tropic_night_count
        integers representing the number of tropical days and tropical nights during the heatwave

    Methods
    -------
    print_heatwave()
    """
    def __init__(self, dates, maxtemps, mintemps):
        """Initializer"""
        self.dates = dates
        self.maxtemps = maxtemps
        self.mintemps = mintemps
        self.year = dates[0][0]
        self.hw_dict = dict() # Dictionary relating dates to TX values
        self.length = len(maxtemps)
        self.tropic_day_count = 0 # Number of tropical days (TX >= 30°)
        self.tropic_night_count = 0 # Number of tropical nights (TN >= 20°)

        # Compute and set the nr. of tropical days/nights
        for i, temp in enumerate(maxtemps):
            if temp >= 30:
                self.tropic_day_count += 1
            if mintemps[i] >= 20:
                self.tropic_night_count += 1
                
        # Fill up the date-TX dictionnary
        for i, date in enumerate(dates):
            self.hw_dict[date] = maxtemps[i]
            
    def print_heatwave(self):
        """Function for printing out relevant info on the heatwave instance"""
        # Set start and end date
        start_date = str(self.dates[0][0]) + '/' + str(self.dates[0][1]) + '/' + str(self.dates[0][2])
        end_date = str(self.dates[-1][0]) + '/' + str(self.dates[-1][1]) + '/' + str(self.dates[-1][2])
        
        print('--' * 30)
        print("HW from " + start_date + ' - ' + end_date)
        print(f"length: {self.length}")
        print(f"tropic days: {self.tropic_day_count}")
        print(f"tropic nights: {self.tropic_night_count}")
        print('--' * 30)

In [8]:
class Dataset:
    """class used for automatic preprocessing of a textfile containing daily temp. information as y | m | d | TX | TN
    Attributes
    ----------
    year, month, day
        lists of integers representing the first three columns containing the year, month (1-12) and day (1-31)
    Tmax, Tmin, T
        lists of floats representing the daily max. temperatures (TX), min. temperatures (TN) and average temperatures (T)
    yearset
        list of integers containing the years covered by the dataset (no duplicates, chronological)
    T_year
        list of integers containing the average yearly temperature for each year covered by the dataset
    T_avg_10yrs
        list of integers containing the 10 year running average temperatures for years covered by the dataset excluding the first 4 and last 6 years
    heatwaves
        list of Heatwave instances representing the heatwaves contained within the dataset (RMI definition)
    hw_day_count, tropic_day_count, tropic_night_count
        dictionaries (int -> int) with years as keys and total number of heatwave days, trop. days and trop. nights as values
    CDD
        dictionary (int -> float) with years as keys and HI values (in cooling degree days) as values
    Tmax_year
        list of floats representing the peak temperature for every year in yearset (chronological)
    days_LH
        dictionary (int -> int) relating every year in yearset to the length of its longest heatwave (in nr. of days)
    Tmax_adj, Tmin_adj
        lists of floats representing adjusted/scaled daily max. temperatures (TX) and min. temperatures (TN) so original and scaled series can be processed
        with one instance of Dataset
    Methods
    -------
    set_T(adjusted=False)
    set_adjusted(adjusted_max, adjusted_min)
    set_avg_temps()
    find_heatwaves(yr, adjusted=False)
    set_heatwaves(adjusted=False)
    get_date(index)
    get_datestring(index)
    set_heatwave_day_counts()
    set_tropics(adjusted=False)
    get_max_temp(yr, adjusted=False)
    set_Tmax_year(adjusted=False)
    set_CDD(adjusted=False)
    set_all(adjusted=False)
    remove_nonexistant_dates()
    """
    def __init__(self, year, month, day, Tmax, Tmin):
        """Initializer"""
        self.year = year
        self.month = month
        self.day = day
        self.Tmax = Tmax
        self.Tmin = Tmin
        self.T = list()
        
        self.yearset = list(set(year)) # Remove duplicates
        self.yearset.sort() # Sort for chronological order
        
        
        self.T_year = list() 
        self.T_avg_10yrs = dict()
        
        self.heatwaves = list() # List of heatwaves
        self.hw_count = dict()
        self.hw_day_count = dict()
        self.tropic_day_count = dict()
        self.tropic_night_count = dict()
        self.CDD = dict()
        self.Tmax_year = list()
        self.days_LH = dict()

        self.Tmax_adj = list() # Adjusted max. temp. list
        self.Tmin_adj = list()
        
        for yr in self.yearset:
            # Initialize the dicts
            self.hw_count[yr] = 0
            self.hw_day_count[yr] = 0
            self.tropic_day_count[yr] = 0
            self.tropic_night_count[yr] = 0
            self.CDD[yr] = 0
    
    def set_T(self, adjusted=False):
        """Setter for daily temperature using original or scaled series (as indicated by the Boolean 'adjusted', default False)"""
        if adjusted:
            tempmax, tempmin = self.Tmax_adj, self.Tmin_adj
        else:
            tempmax, tempmin = self.Tmax, self.Tmin

        # Compute avg. daily temp. as the arithmetic mean of TX and TN
        self.T = [(tempmax[i] + tempmin[i]) * 0.5 for i in range(len(tempmax))]
        
    def set_adjusted(self, adjusted_max, adjusted_min):
        """Setter used to pass scaled series to the Dataset instance"""
        self.Tmax_adj, self.Tmin_adj = adjusted_max, adjusted_min
        
    def set_avg_temps(self):
        """Compute yearly averages and 10yr averages"""
        yr = self.year[0]
        T_year, temperatures = list(), list()
        for i in range(len((self.year))):
            # Collect all temperatures of each year in respective lists and take the mean
            if self.year[i] == yr:
                if i == (len(self.year)-1): # Last entry is reached
                    yeartemp = np.mean(temperatures)
                    T_year.append(yeartemp)
                    break
                temperatures.append(self.T[i])

            else:
                yeartemp = np.mean(temperatures)
                T_year.append(yeartemp)
                temperatures = [self.T[i]]
                yr = self.year[i]

        # Compute and store the 10 year running averages
        T_avg_10yrs = dict()
        for i in range(4, len(T_year) - 6):
            averaged_temp = np.mean(T_year[i-4:i+6])
            T_avg_10yrs[self.yearset[i]] = averaged_temp

        # Set the attributes
        self.T_year = T_year
        self.T_avg_10yrs = T_avg_10yrs
        
    def find_heatwaves(self, yr, adjusted=False):
        """Determine the index ranges of all heatwaves for a given year, 
        using original or scaled series (as indicated by the Boolean 'adjusted', default False)"""
        # Use scaled temperature series if needed
        if adjusted:
            templist = self.Tmax_adj
        else:
            templist = self.Tmax
            
        yr_index = list(self.year).index(yr)
        
        # To avoid complications involving leap years, assume heatwaves only take place between March and November
        march_index = list(self.month).index(3, yr_index)
        november_index = list(self.month).index(11, yr_index)
        
        yr_maxtemps = list() # List for storing TX values between March and November
        i=0
        while (march_index + i < len(self.year) and self.year[march_index + i] == yr and i < november_index):
            yr_maxtemps.append(templist[march_index + i])
            i += 1

        heatwave_ranges = find_consecutive_highs(yr_maxtemps) # Call external function for detecting heatwaves in daily temperature series
        for j, hw in enumerate(heatwave_ranges):
            heatwave_ranges[j] = (hw[0] + march_index, hw[1] + march_index) # Adjust the ranges relative to March 1st
        return heatwave_ranges
    
    def set_heatwaves(self, adjusted=False):
        """Find and set all heatwaves in the Dataset,
        using original or scaled series (as indicated by the Boolean 'adjusted', default False)"""
        # Use adjusted temperature list if needed
        if adjusted:
            templist = self.Tmax_adj
            min_templist = self.Tmin_adj
        else:
            templist = self.Tmax
            min_templist = self.Tmin
        
        for yr in self.yearset:
            max_len = 0
            heatwave_ranges = self.find_heatwaves(yr, adjusted)
            self.hw_count[yr] = len(heatwave_ranges) # Also set the number of heatwaves
            for hw in heatwave_ranges:
                start_i, end_i = hw[0], hw[1]+1
                dates = [self.get_date(i) for i in range(start_i, end_i)]
                maxtemps = [templist[i] for i in range(start_i, end_i)]
                mintemps = [min_templist[i] for i in range(start_i, end_i)]
                self.heatwaves.append(Heatwave(dates, maxtemps, mintemps))
                if len(dates) > max_len:
                    max_len = len(dates) # Keep track of longest heatwave for every year
            self.days_LH[yr] = max_len
        
    
    def get_date(self, index):
        """Obtain date by index"""
        y, m, d = self.year[index], self.month[index], self.day[index]
        return (y, m, d)
        
    def get_datestring(self, index):
        """Obtain a string representation of the date by a given index"""
        m, d = self.month[index], self.day[index]
        return str(int(d)) + '/' + str(int(m))
    
    def set_heatwave_day_counts(self):
        """Set the nr. of heatwave days"""
        for hw in self.heatwaves:
            yr = hw.year
            self.hw_day_count[yr] += hw.length
    
    def set_tropics(self, adjusted=False):
        """Set the nr. of tropical days and nights,
        using original or scaled series (as indicated by the Boolean 'adjusted', default False)"""
        # Use adjusted temperature list if needed
        if adjusted:
            max_templist = self.Tmax_adj
            min_templist = self.Tmin_adj
        else:
            max_templist = self.Tmax
            min_templist = self.Tmin
        
        for i, yr in enumerate(self.year): # Enumerate over every day of every year
            if max_templist[i] >= 30:
                self.tropic_day_count[yr] += 1
            if min_templist[i] >= 20:
                self.tropic_night_count[yr] += 1
           
                
    def get_max_temp(self, yr, adjusted=False):
        """Determine max. temp. for a given year,
        using original or scaled series (as indicated by the Boolean 'adjusted', default False)"""
        # Use adjusted temperature list if needed
        if adjusted:
            templist = self.Tmax_adj
        else:
            templist = self.Tmax
        indices = np.where(np.asarray(self.year) == yr)[0]
        return np.max(templist[indices[0]:indices[-1]])
    
    def set_Tmax_year(self, adjusted=False):
        """Set the maximal yearly temps"""
        self.Tmax_year = []
        for yr in self.yearset:
            maxtemp = self.get_max_temp(yr, adjusted)
            self.Tmax_year.append(maxtemp)
            
    def set_CDD(self, adjusted=False):
        """Set the CDD dictionary"""
        # Use adjusted temperature list if needed
        if adjusted:
            templist = self.Tmax_adj
        else:
            templist = self.Tmax
            
        for i,yr in enumerate(self.year):
            count = templist[i] - 25.0
            if count > 0:
                self.CDD[yr] += count
    
    def set_all(self, adjusted=False):
        """Encompassing function that executes all setters"""
        # first clear heatwaves
        self.heatwaves = []
        
        self.remove_nonexistant_dates()
        
        self.set_T(adjusted)
        self.set_avg_temps()
        self.set_heatwaves(adjusted)
        self.set_heatwave_day_counts()
        self.set_Tmax_year(adjusted)
        self.set_CDD(adjusted)
        self.set_tropics(adjusted)
            
    def plot_heatwaves(self, ax, title, adjusted=False):
        """Plot an overview of all heatwaves"""
        # Use adjusted temperature list if needed
        if adjusted:
            templist = self.Tmax_adj
        else:
            templist = self.Tmax
            
        hw_matrix = np.zeros((len(self.yearset), 245))
        for i, yr in enumerate(self.yearset):
            yr_index = list(self.year).index(yr)
            march_index = list(self.month).index(3, yr_index)
            heatwave_ranges = self.find_heatwaves(yr, adjusted)
        
            for hw in heatwave_ranges:
                for j in range(hw[0], hw[1]+1):
                    hw_matrix[i][j-march_index] = templist[j]

        yticks = list(range(0, len(self.yearset), 10))
        ylabels = list()
        for i in yticks:
            ylabels.append(str(int(self.yearset[i])))

        xticks = [0, 31, 61, 92, 122, 153, 184, 214, 245]
        xlabels = list()
        for i in xticks:
            xlabels.append(self.get_datestring(march_index + i))


        ax = sns.heatmap(hw_matrix, vmin=20., vmax=50., cmap="coolwarm", cbar_kws={'label': 'Temp. (°C)'})
        ax.set_yticks(yticks)
        ax.set_yticklabels(ylabels)
        ax.set_xticks(xticks)
        ax.set_xticklabels(xlabels)
        ax.set(title=title) 
        
    def remove_nonexistant_dates(self):
        """Function to remove NaN entries from the dataset"""
        nan_indices = np.where(np.isnan(self.Tmax))[0]
        self.year = np.delete(self.year, nan_indices)
        self.month = np.delete(self.month, nan_indices)
        self.day = np.delete(self.day, nan_indices)
        self.Tmax = np.delete(self.Tmax, nan_indices)
        self.Tmin = np.delete(self.Tmin, nan_indices)
    
        
        
